<a href="https://colab.research.google.com/github/WaelRabah/TT_LLM/blob/main/TT_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TT_LLM: Tensor-Train Compression for LLM Linear Layers

This notebook compresses the linear layers of a HuggingFace causal LM using
**Tensor-Train (TT) decomposition** with two faithful weight initialisations
(`svd` and `tt_svd`) and one ablation (`random`).

All reusable code lives in the `tt_llm` package (`tt_llm/layers.py`, 
`tt_llm/decompositions.py`, `tt_llm/compress.py`); this notebook only handles
Colab interaction: installs, HF login, model loading, inference and comparison.

## Section 1: Environment Setup

Install the `tt_llm` package directly from the GitHub repo (works in Colab) and
the libraries required to load the model.

In [ ]:
%pip install -q transformers accelerate huggingface_hub

# The TT_LLM repo is private; authenticate the install with a GitHub token
# stored in Colab Secrets (key: GITHUB_TOKEN). Fall back to a public clone
# if the token is unavailable.
try:
    from google.colab import userdata
    _gh_token = userdata.get('GITHUB_TOKEN')
    _install_url = f"git+https://{_gh_token}@github.com/WaelRabah/TT_LLM.git"
except Exception:
    _gh_token = None
    _install_url = "git+https://github.com/WaelRabah/TT_LLM.git"

%pip install -q "{_install_url}"

## Section 2: HuggingFace Login
Retrieve the HF token from Colab Secrets (optional; only needed for gated
models).

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=hf_token)
except Exception as e:
    print("HF_TOKEN not found in Secrets. Set it for gated models or use a public model.")

## Section 3: Load the Model and Tokenizer

We use `HuggingFaceTB/SmolLM2-360M` (a small open model). The same flow works
for other `AutoModelForCausalLM` checkpoints.

In [ ]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM

if 'model' in globals():
    del model
torch.cuda.empty_cache()
gc.collect()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Target execution device: {device}")

model_id = "HuggingFaceTB/SmolLM2-360M"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map=device,
)
print(f"Model and tokenizer loaded! Model is on: {model.device}")

## Section 4: Baseline Inference

Run a prompt on the uncompressed model to verify it produces coherent text.

In [ ]:
prompt = "Explain the concept of quantum computing in one simple sentence."

def run_inference(model, prompt, **gen_kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=50, do_sample=True,
                                  temperature=0.7, top_p=0.9, **gen_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- Baseline Inference (uncompressed) ---")
print("Prompt:", prompt)
print("Response:", run_inference(model, prompt))

## Section 5: Uniform Compression (SVD / TT-SVD / Random)

Recursively swap every `nn.Linear` in the model with a TT-decomposed layer.
- `init_method='svd'`: global best-rank SVD approximation of each weight
  matrix, embedded as TT cores.
- `init_method='tt_svd'`: canonical TT-SVD with per-unfolding truncation.
- `init_method='random'`: random TT cores (ablation; reproduces the original
  notebook's broken behaviour).

`compression_pct=10` means cut parameters by 10% (keep 90%).

In [ ]:
from tt_llm import compress_model_inplace

print("--- Compressing with SVD init (10% reduction) ---")
compress_model_inplace(model, compression_pct=10, layer_type='linear', init_method='svd')
print("\nResponse after SVD-init compression:")
print(run_inference(model, prompt))

## Section 6: TT-SVD Init — Reload and Recompress

Reload the fresh model (TT-SVD is destructive; for an apples-to-apples
comparison we start from the original weights each time).

In [ ]:
del model
torch.cuda.empty_cache()
gc.collect()

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map=device,
)

print("--- Compressing with TT-SVD init (10% reduction) ---")
compress_model_inplace(model, compression_pct=10, layer_type='linear', init_method='tt_svd')
print("\nResponse after TT-SVD-init compression:")
print(run_inference(model, prompt))

## Section 7: Targeted (Activation-Aware) Compression

Instead of compressing every layer by the same amount, we:
1. Run a set of calibration prompts through the model with forward hooks on
   every `nn.Linear`.
2. Score each layer's *importance* = Frobenius norm of its input activations.
3. Allocate the global parameter budget proportional to importance —
   important layers keep more parameters (less compression), less-important
   layers are compressed more aggressively.

This should preserve output quality better than uniform compression at the
same overall parameter budget.

In [ ]:
from tt_llm import compress_model_targeted

del model
torch.cuda.empty_cache(); gc.collect()
model = AutoModelForCausalLM.from_pretrained(
    model_id, dtype=torch.bfloat16 if device == "cuda" else torch.float32, device_map=device,
)

calibration_prompts = [
    "Explain the concept of quantum computing in one simple sentence.",
    "Write a short story about a robot learning to paint.",
    "What are the main causes of climate change?",
    "Describe the process of photosynthesis.",
    "How does a transformer neural network work?",
]

print("--- Targeted compression (10% reduction, SVD init) ---")
compress_model_targeted(
    model, tokenizer, calibration_prompts,
    compression_pct=20, importance_cutoff=0.2, layer_type='linear', init_method='svd',
    max_length=128, device=device
)
print("\nResponse after targeted compression:")
print(run_inference(model, prompt))

In [ ]:
print("\nResponse after targeted compression:")
print(run_inference(model, prompt))

## Section 8: Random-Init Ablation

The original notebook's behaviour: TT cores are random, so the compressed
model produces garbage. We keep it for comparison.

In [ ]:
del model
torch.cuda.empty_cache()
gc.collect()

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map=device,
)

print("--- Compressing with RANDOM init (ablation) ---")
compress_model_inplace(model, compression_pct=10, layer_type='linear', init_method='random')
print("\nResponse after random-init compression (expect garbage):")
print(run_inference(model, prompt))

## Section 9: SoLA — Soft Activation Sparsity + Low-Rank Decomposition

SoLA (Huang et al., AAAI 2025) is a training-free compression method that
combines two ideas:

1. **Activation-aware weighted SVD**: Instead of plain SVD on the weight
   matrix, the input dimension is scaled by per-feature activation norms
   (captured via calibration prompts). This ensures that directions with
   large activations are preserved in the low-rank approximation.

2. **Adaptive per-layer rank allocation**: Each  receives a
   different truncation rank proportional to its activation energy
   (Frobenius norm of input activations), so important layers are
   compressed less.

3. **Soft activation sparsity** (FFN only): In feed-forward 
   layers, the top-k most important intermediate neurons (by activation
   energy) are kept dense; only the rest are low-rank decomposed.

The  parameter controls the fraction of neurons kept dense
(1.0 = pure low-rank, 0.5 = keep top-50% dense).


In [ ]:
from tt_llm import compress_model_sola

del model
torch.cuda.empty_cache(); gc.collect()
model = AutoModelForCausalLM.from_pretrained(
    model_id, dtype=torch.bfloat16 if device == "cuda" else torch.float32, device_map=device,
)

calibration_prompts = [
    "Explain the concept of quantum computing in one simple sentence.",
    "Write a short story about a robot learning to paint.",
    "What are the main causes of climate change?",
    "Describe the process of photosynthesis.",
    "How does a transformer neural network work?",
]

print("--- SoLA compression (30% reduction, sparsity_ratio=0.5) ---")
compress_model_sola(
    model, tokenizer, calibration_prompts,
    compression_pct=30, sparsity_ratio=0.5,
    max_length=128, device=device,
)
print("
Response after SoLA compression:")
print(run_inference(model, prompt))

## Section 10: Method Comparison

| Method | Init | Activation-Aware | Rank Allocation | TT Cores |
|--------|------|-----------------|-----------------|----------|
| Uniform SVD |  | No | Uniform | Yes (rank-r as TT) |
| Uniform TT-SVD |  | No | Uniform | Yes (canonical TT-SVD) |
| Targeted |  | Yes (Frobenius) | Energy-proportional | Yes |
| **SoLA** | ASVD | **Yes (per-feature)** | **Energy-proportional** | No (pure SVD) |
| Random |  | No | Uniform | Yes |

| Method | Compression | Granularity | FFN Sparsity | Training-Free |
|--------|------------|-------------|--------------|---------------|
| TT-LLM (targeted) | TT-rank budget | Per-layer | No | Yes |
| **SoLA** | SVD-rank budget | Per-layer + per-feature | **Yes** | Yes |

**Key difference**: TT-LLM uses Tensor-Train decomposition (multi-core,
higher-order compression), while SoLA uses standard SVD with
activation-aware weighting. SoLA's soft activation sparsity for FFN
layers is a unique feature — it keeps the most important neurons dense
rather than compressing everything uniformly.
